In [1]:
# =========================================
# Cell 1 — Install dependencies
# =========================================
!pip -q install google-genai>=1.0.0 datasets>=2.19.0 pandas>=2.2.0 numpy>=1.26.0 scikit-learn>=1.3.0 tqdm>=4.66.0 matplotlib>=3.8.0


In [2]:
# =========================================
# Cell 2 — Keys (type/paste in)
# =========================================
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY (hidden): ")
os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN (required for gated FPB): ").strip()

assert os.environ.get("GEMINI_API_KEY"), "Missing GEMINI_API_KEY"
assert os.environ.get("HF_TOKEN"), "Missing HF_TOKEN (FPB is gated)"
print("✅ Keys set.")


Enter GEMINI_API_KEY (hidden): ··········
Enter HF_TOKEN (required for gated FPB): ··········
✅ Keys set.


In [3]:
# =========================================
# Cell 3 — Load FPB dataset from Hugging Face
# =========================================
from datasets import load_dataset

DATASET_NAME = "TheFinAI/en-fpb"
SPLIT = "test"   # try "train"/"validation"/"test" depending on what's available

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ["HF_TOKEN"])
print(ds)
print("Columns:", ds.column_names)
print("Example row:\n", ds[0])


README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example row:
 {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


In [4]:
# =========================================
# Cell 4 — Gemini 2.5 Flash: prompt + API helpers
# =========================================
import re, time
from google import genai
from google.genai import types

MODEL_ID = "gemini-2.5-flash"  # :contentReference[oaicite:3]{index=3}

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def build_prompt(text: str, choices):
    return (
        "Classify the sentiment of the following financial text.\n"
        "Respond ONLY with one label from Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(map(str, choices))}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if not model_text:
        return None
    s = model_text.strip().lower()

    # exact match
    for c in choices:
        if s == str(c).strip().lower():
            return str(c)

    # whole-word match (handles extra words)
    for c in choices:
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def gemini_predict(prompt: str, model: str = MODEL_ID, max_retries: int = 6, sleep_base: float = 1.0):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=8,
                ),
            )

            text_out = getattr(resp, "text", None)

            # usage metadata (make JSON-safe if present)
            usage = getattr(resp, "usage_metadata", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "prompt_token_count": getattr(usage, "prompt_token_count", None),
                    "candidates_token_count": getattr(usage, "candidates_token_count", None),
                    "total_token_count": getattr(usage, "total_token_count", None),
                }

            meta = {"usage": usage_dict}
            return text_out, meta

        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))

    return None, {"error": last_err}

# ---- Robust row adapter (works if schema is either (text, choices, gold) or (text/sentence, label)) ----
def normalize_example(ex: dict):
    # find text field
    text_key = None
    for k in ["text", "sentence", "content", "input", "news", "headline"]:
        if k in ex:
            text_key = k
            break
    if text_key is None:
        raise KeyError(f"Couldn't find a text field in keys={list(ex.keys())}")

    text = ex[text_key]

    # if TheFinAI "flare-*" style:
    if "choices" in ex and ("gold" in ex or "label" in ex):
        choices = list(ex["choices"])
        gold = int(ex.get("gold", ex.get("label")))
        return text, choices, gold

    # fallback: common FPB labels
    # assume labels are ints 0/1/2 or strings
    choices = ["negative", "neutral", "positive"]
    if "label" in ex:
        lab = ex["label"]
    elif "gold" in ex:
        lab = ex["gold"]
    elif "sentiment" in ex:
        lab = ex["sentiment"]
    else:
        raise KeyError(f"Couldn't find label/gold/sentiment in keys={list(ex.keys())}")

    if isinstance(lab, str):
        lab_norm = lab.strip().lower()
        if lab_norm in choices:
            gold = choices.index(lab_norm)
        else:
            raise ValueError(f"Unknown string label: {lab}")
    else:
        gold = int(lab)
        if gold < 0 or gold >= len(choices):
            raise ValueError(f"Label int {gold} out of range for choices={choices}")

    return text, choices, gold


In [5]:
# =========================================
# Cell 5 — Run evaluation (full split by default) + resume support
# =========================================
import os, json, random, time
from tqdm import tqdm

SEED = 42
MAX_SAMPLES = None         # None = FULL split; set an int to test (e.g., 200)
SHUFFLE = True
SLEEP_BETWEEN_CALLS = 0.25

random.seed(SEED)

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"gemini25flash_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

# resume
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text, choices, gold = normalize_example(ex)

        prompt = build_prompt(text, choices)
        model_out, meta = gemini_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,  # JSON-serializable
        }

        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)


Already completed: 0


Evaluating: 100%|██████████| 970/970 [21:40<00:00,  1.34s/it]

✅ Saved raw responses to: text_responses/gemini25flash_TheFinAI_en-fpb_test.jsonl


In [6]:
# =========================================
# Cell 6 — Compute metrics + save artifacts (loads full jsonl)
# =========================================
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import json

all_rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            all_rows.append(json.loads(line))
        except:
            pass

df = pd.DataFrame(all_rows)

valid_mask = df["predicted_index"] >= 0
labels_sorted = sorted(set(df["gold"].tolist()))

acc_all = accuracy_score(df["gold"], df["predicted_index"])
acc_valid = accuracy_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"]) if valid_mask.any() else None

metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": int(len(df)),
    "num_valid_predictions": int(valid_mask.sum()),
    "accuracy_all": float(acc_all),
    "accuracy_valid_only": (float(acc_valid) if acc_valid is not None else None),
    "f1_macro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="macro", labels=labels_sorted)) if valid_mask.any() else None,
}

cm = confusion_matrix(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted) if valid_mask.any() else None

metrics_path = f"evaluation/{run_tag}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {"metrics": metrics, "confusion_matrix_labels": labels_sorted, "confusion_matrix": (cm.tolist() if cm is not None else None)},
        f, ensure_ascii=False, indent=2
    )

preds_path = f"evaluation/{run_tag}_predictions.csv"
df.to_csv(preds_path, index=False)

print("✅ Metrics saved to:", metrics_path)
print("✅ Predictions saved to:", preds_path)

if valid_mask.any():
    print("\nClassification report (valid predictions only):")
    print(classification_report(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted))
else:
    print("No valid predictions to report.")


✅ Metrics saved to: evaluation/gemini25flash_TheFinAI_en-fpb_test_metrics.json
✅ Predictions saved to: evaluation/gemini25flash_TheFinAI_en-fpb_test_predictions.csv

Classification report (valid predictions only):
              precision    recall  f1-score   support

           0       0.74      0.87      0.80        30
           1       0.83      0.58      0.68        26
           2       0.83      1.00      0.91        15

    accuracy                           0.79        71
   macro avg       0.80      0.81      0.80        71
weighted avg       0.80      0.79      0.78        71

